# AI-Powered Business Intelligence

**CODTECH IT Solutions — Data Analytics Internship — Task 3**

Intern: Chakiri Subramanyam | Intern ID: CITS5137

## 1. Generate the Dataset

In [ ]:
"""
generate_data.py
Generates a synthetic business dataset for the AI-Powered Business
Intelligence task (Codtech IT Solutions - Data Analytics Internship, Task 3).
"""

import numpy as np
import pandas as pd

np.random.seed(42)

N = 2000
date_range = pd.date_range(start="2023-01-01", end="2024-12-31", periods=N)

departments = ["Sales", "Marketing", "Operations", "HR", "Finance", "IT"]
regions = ["North", "South", "East", "West"]
products = ["Product A", "Product B", "Product C", "Product D", "Product E"]

dept = np.random.choice(departments, N)
region = np.random.choice(regions, N)
product = np.random.choice(products, N)

# Revenue influenced by department and product
base_revenue = {"Product A": 500, "Product B": 700, "Product C": 300,
                "Product D": 900, "Product E": 450}
revenue = np.array([
    base_revenue[p] * np.random.uniform(0.7, 1.5) +
    np.random.normal(0, 50)
    for p in product
])
revenue = np.clip(revenue, 50, 2000).round(2)

cost = (revenue * np.random.uniform(0.4, 0.75, N)).round(2)
profit = (revenue - cost).round(2)

employee_count = np.random.randint(5, 80, N)
customer_satisfaction = np.clip(np.random.normal(3.8, 0.6, N), 1, 5).round(1)
marketing_spend = np.clip(np.random.normal(200, 80, N), 20, 600).round(2)
units_sold = np.random.randint(10, 500, N)

df = pd.DataFrame({
    "Date": date_range,
    "Department": dept,
    "Region": region,
    "Product": product,
    "Revenue": revenue,
    "Cost": cost,
    "Profit": profit,
    "Units_Sold": units_sold,
    "Employee_Count": employee_count,
    "Marketing_Spend": marketing_spend,
    "Customer_Satisfaction": customer_satisfaction,
})

df.to_csv("data/business_data.csv", index=False)
print(f"Generated {len(df):,} rows -> data/business_data.csv")


## 2. AI Analysis — Clustering, Forecasting, Anomaly Detection & Visualizations

In [ ]:
"""
ai_business_intelligence.py
Task 3: AI-Powered Business Intelligence
Codtech IT Solutions - Data Analytics Internship

Loads the business dataset, performs AI-powered analysis including clustering,
revenue forecasting, and anomaly detection, then saves visualizations to output/.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "data/business_data.csv"
OUT_DIR = "output"

# ---------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)
print("Shape:", df.shape)
print(df.head())

# ---------------------------------------------------------------
# 2. Revenue trend over time
# ---------------------------------------------------------------
monthly = df.groupby("Month")[["Revenue", "Cost", "Profit"]].sum().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(monthly["Month"], monthly["Revenue"], label="Revenue", marker="o", linewidth=2)
plt.plot(monthly["Month"], monthly["Cost"], label="Cost", marker="s", linewidth=2)
plt.plot(monthly["Month"], monthly["Profit"], label="Profit", marker="^", linewidth=2)
plt.title("Monthly Revenue, Cost & Profit Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Amount ($)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_revenue_cost_profit_trend.png")
plt.close()

# ---------------------------------------------------------------
# 3. Revenue by Department
# ---------------------------------------------------------------
dept_rev = df.groupby("Department")["Revenue"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=dept_rev.index, y=dept_rev.values, hue=dept_rev.index,
            palette="Blues_d", legend=False)
plt.title("Total Revenue by Department", fontsize=14, fontweight="bold")
plt.xlabel("Department")
plt.ylabel("Total Revenue ($)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_revenue_by_department.png")
plt.close()

# ---------------------------------------------------------------
# 4. Profit Margin by Product
# ---------------------------------------------------------------
product_df = df.groupby("Product")[["Revenue", "Profit"]].sum()
product_df["Profit_Margin_%"] = (product_df["Profit"] / product_df["Revenue"] * 100).round(2)
product_df = product_df.sort_values("Profit_Margin_%", ascending=False).reset_index()

plt.figure(figsize=(9, 5))
sns.barplot(data=product_df, x="Product", y="Profit_Margin_%", hue="Product",
            palette="Greens_d", legend=False)
plt.title("Profit Margin % by Product", fontsize=14, fontweight="bold")
plt.xlabel("Product")
plt.ylabel("Profit Margin (%)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_profit_margin_by_product.png")
plt.close()

# ---------------------------------------------------------------
# 5. AI: Customer Segmentation via K-Means Clustering
# ---------------------------------------------------------------
cluster_features = df[["Revenue", "Units_Sold", "Marketing_Spend", "Customer_Satisfaction"]].copy()
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(cluster_scaled)
score = silhouette_score(cluster_scaled, df["Cluster"])
print(f"Silhouette Score: {score:.3f}")

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Revenue", y="Customer_Satisfaction",
                hue="Cluster", palette="Set1", alpha=0.6)
plt.title(f"AI Customer Segmentation — K-Means Clustering\n(Silhouette Score: {score:.3f})",
          fontsize=13, fontweight="bold")
plt.xlabel("Revenue ($)")
plt.ylabel("Customer Satisfaction")
plt.legend(title="Cluster")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_customer_segmentation.png")
plt.close()

# ---------------------------------------------------------------
# 6. AI: Revenue Forecasting (Linear Regression)
# ---------------------------------------------------------------
monthly["TimeIndex"] = np.arange(len(monthly))
X_time = monthly[["TimeIndex"]]
y_rev = monthly["Revenue"]

reg = LinearRegression()
reg.fit(X_time, y_rev)
monthly["Predicted"] = reg.predict(X_time)

# Forecast next 6 months
future_idx = np.arange(len(monthly), len(monthly) + 6).reshape(-1, 1)
future_months = pd.date_range(monthly["Month"].max(), periods=7, freq="MS")[1:]
future_pred = reg.predict(future_idx)

plt.figure(figsize=(12, 5))
plt.plot(monthly["Month"], monthly["Revenue"], label="Actual Revenue", marker="o", linewidth=2)
plt.plot(monthly["Month"], monthly["Predicted"], linestyle="--", label="Trend Line", linewidth=2)
plt.plot(future_months, future_pred, linestyle=":", marker="^", color="red",
         label="Forecast (Next 6 Months)", linewidth=2)
plt.axvline(monthly["Month"].max(), color="gray", linestyle="--", alpha=0.5)
plt.title("AI Revenue Forecasting — Linear Regression", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Revenue ($)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_revenue_forecast.png")
plt.close()

# ---------------------------------------------------------------
# 7. AI: Anomaly Detection (Isolation Forest)
# ---------------------------------------------------------------
iso = IsolationForest(contamination=0.05, random_state=42)
df["Anomaly"] = iso.fit_predict(df[["Revenue", "Profit", "Marketing_Spend"]])
df["Anomaly_Label"] = df["Anomaly"].map({1: "Normal", -1: "Anomaly"})

plt.figure(figsize=(10, 6))
colors = {"Normal": "#4c72b0", "Anomaly": "#dd4444"}
for label, grp in df.groupby("Anomaly_Label"):
    plt.scatter(grp["Revenue"], grp["Profit"], label=label,
                color=colors[label], alpha=0.5, s=20)
plt.title("AI Anomaly Detection — Isolation Forest\n(Revenue vs Profit)",
          fontsize=13, fontweight="bold")
plt.xlabel("Revenue ($)")
plt.ylabel("Profit ($)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_anomaly_detection.png")
plt.close()

# ---------------------------------------------------------------
# 8. Heatmap: Region vs Department Revenue
# ---------------------------------------------------------------
pivot = df.pivot_table(index="Region", columns="Department",
                       values="Revenue", aggfunc="sum")

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.3)
plt.title("Revenue Heatmap: Region vs Department", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/07_region_department_heatmap.png")
plt.close()

# ---------------------------------------------------------------
# 9. Save summary
# ---------------------------------------------------------------
summary = pd.DataFrame({
    "Total_Revenue": [df["Revenue"].sum().round(2)],
    "Total_Profit": [df["Profit"].sum().round(2)],
    "Avg_Customer_Satisfaction": [df["Customer_Satisfaction"].mean().round(2)],
    "Anomalies_Detected": [(df["Anomaly"] == -1).sum()],
    "Silhouette_Score": [round(score, 3)],
    "Top_Department": [dept_rev.idxmax()],
    "Top_Product_Margin": [product_df.iloc[0]["Product"]],
})
summary.to_csv(f"{OUT_DIR}/summary_stats.csv", index=False)

print("\nAll charts saved to output/")
print(summary.T)
